## 0. Kernel setup (run in a terminal, not in this notebook)

This notebook reuses the same `2ndWorkshop` conda environment and kernel as
`Workshop2_Part2a_Server.ipynb` -- if you've already set that up and run the install
cell there, you can select the same `Python (2ndWorkshop)` kernel here and skip
straight to C1.

**Prerequisite:** `Workshop2_Part2a_Server.ipynb` must already be running (in a separate
kernel) before you run the cells below -- this notebook connects to the tools it
exposes over MCP.

# Workshop 2, Part 2b: LangGraph Agent (Boilermaker TA) — ReAct Pattern

This is the **agent half** of Part 2. It connects to the MCP tool server from
`Workshop2_Part2a_Server.ipynb` over HTTP and orchestrates its tools using the classic
**ReAct** pattern: a single generic `agent` node, bound to every tool at once, looping
with a `ToolNode` until the model stops requesting tools. That's the Boilermaker TA.

**Before running this notebook:** start `Workshop2_Part2a_Server.ipynb` in a separate
kernel and leave it running.

**What you'll do:**
- Connect to the MCP server and load every tool it exposes
- Build a generic `agent` ⇄ `tools` loop, with the system prompt (C5) as the only thing
  telling the model to look things up before writing a notification -- nothing in the
  code stops it from calling `create_notification` on its very first turn
- Run the Boilermaker TA on a real task and see whether the model gets the order right
  on its own

**LLM backend:** same default as Workshop 1 and Part 1, a small local HuggingFace
model, no API key or external server required. Tool-calling reliability -- and how
reliably the model follows the ordering rules in the prompt -- drops with model size,
so **C2** also shows how to switch to Ollama, Purdue GenAI, Anthropic, or OpenAI with
one line if the local model struggles.

**Where prompting alone falls short:** because the only thing enforcing order here is
the system prompt (C5), a weak enough model can still call `create_notification` before
it has looked anything up, or write a placeholder date like `[insert date]` into
`announcements.txt`. This is a real, observed failure mode with the local default model
-- not a strawman. If you want to rule that failure out structurally instead of hoping
the prompt holds, **`Workshop2_Part2b_Agent_hardgraph.ipynb`** rebuilds this exact task
as a **fixed two-node graph** (gather, then write) where the model physically cannot
call `create_notification` first. Same tools, same task, same system-prompt idea -- but
the constraint moves from prompt text into the graph's structure. Worth trying once
you've run this notebook, especially with the weaker local model.

**Note:** this notebook writes to `workshop_outputs/announcements.txt`, the same file
`Workshop2_Part2b_Agent_hardgraph.ipynb` writes to. Clear it between runs if you want a
clean read on which version produced what.

## 0. Install dependencies

Run once, then restart the kernel. Same packages as `Workshop2_Part2b_Agent_hardgraph.ipynb`
-- skip this if you already ran it there in the same environment.

In [1]:
! pip install torch transformers accelerate langchain-huggingface langchain langchain-core langgraph langchain-mcp-adapters langchain-ollama python-dotenv


## C1. Imports for the agent

In [2]:
import os
import asyncio
import logging
from pathlib import Path
from typing import Annotated
from typing_extensions import TypedDict

from dotenv import load_dotenv
from transformers import pipeline
from langchain.chat_models import init_chat_model
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv()

BASE_DIR = Path(".").resolve()
ANNOUNCEMENTS_FILE = BASE_DIR / "workshop_outputs/announcements.txt"   # shared with Part 2b

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


/Users/elhambarezi/miniconda3/envs/2ndWorkshop/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## C2. Configure the LLM backend

Everything is set with plain variables in the next cell — no environment variables
needed (API keys still go in `.env`). Five presets are shown; uncomment the ONE you
want and re-run the cell.

**Tradeoff to know:** tool-calling reliability scales with model size/quality.
Qwen2.5-Instruct models support function calling even at 0.5B, but a 0.5B model will
miss or malform tool calls more often than a hosted API -- and since this notebook's
only ordering guarantee *is* tool-calling reliability plus the system prompt (C5), a
weak model here is doubly likely to get the order wrong. If the agent seems to ignore
its tools (e.g. it never writes to `announcements.txt`), or calls `create_notification`
before looking anything up, switch presets.

- **Local HF (default)** — runs anywhere (local or server), no server or API key needed,
  but weakest tool-calling reliability.
- **Ollama** — free, and much better tool calling than the local default.

  **If you have sudo/root (your own Mac or Linux machine)**, just use the official
  installer — no need for the manual steps below:
  ```bash
  # macOS
  brew install ollama          # or download the app from https://ollama.com/download

  # Linux, with sudo
  curl -fsSL https://ollama.com/install.sh | sh
  ```
  Then `ollama serve` (it may already be running as a background service after install)
  and `ollama pull llama3.2` or any other ollama models you want.

  **On Gilbreth (or any shared cluster without sudo)**, the official installer won't
  work -- it needs root to place the binary and register a service. Use this no-sudo
  userspace install instead (a static binary extracted straight into your home
  directory, no root required). **Run every command below in a terminal opened from
  inside Jupyter** (JupyterLab's File > New > Terminal, or Open OnDemand's Jupyter app
  terminal button) rather than a separate `ssh` session, a Jupyter-launched terminal
  runs on the exact same compute node as your notebook kernel, so `ollama serve`
  started there is immediately reachable at `127.0.0.1` from this notebook, no extra
  setup needed. (A fresh `ssh` login can land you on a different node, where
  `127.0.0.1` means something else entirely, that's the usual reason a notebook
  "can't find" a server that looks perfectly fine in that terminal.)
  ```bash
  # 1. Point Ollama's model storage at scratch instead of home. Home quotas on
  #    Gilbreth are small and model weights are multi-GB -- symlinking this now avoids
  #    filling your home quota before you even pull a model (the same quota exhaustion
  #    that can make create_notification fail to write announcements.txt once home is full).
  mkdir -p /scratch/gilbreth/$USER/.ollama/models
  mkdir -p ~/.ollama
  ln -s /scratch/gilbreth/$USER/.ollama/models ~/.ollama/models

  # 2. Download the static binary (no root required) and unpack into ~/bin
  mkdir -p ~/bin
  curl -L https://ollama.com/download/ollama-linux-amd64.tar.zst -o ~/ollama-linux-amd64.tar.zst
  tar --use-compress-program=zstd -xvf ~/ollama-linux-amd64.tar.zst -C ~/bin

  # 3. Add the extracted binary to PATH -- the archive unpacks its own bin/ subfolder,
  #    so the executable ends up at ~/bin/bin/ollama, not ~/bin/ollama
  echo 'export PATH=$PATH:~/bin/bin' >> ~/.bashrc
  source ~/.bashrc

  # 4. Verify the install, then start the server and pull a model
  ollama --version
  ollama serve &            # backgrounds it in this terminal; keep the terminal/session
                             # alive for as long as you need the server running
  ollama pull llama3.2       # once `ollama serve` is up
  ```
  **"address already in use"?** Another process (often a leftover `ollama serve` from a
  previous session, sometimes another user's on a shared node) already has port 11434.
  Start on a different port instead:
  ```bash
  OLLAMA_HOST=127.0.0.1:11435 ollama serve &
  OLLAMA_HOST=127.0.0.1:11435 ollama pull llama3.2
  ```
  If you do this, `LLM_BASE_URL` in the preset below can no longer be `None` -- it must
  point at the same port, e.g. `LLM_BASE_URL = "http://127.0.0.1:11435"`, or the
  LangChain client will keep talking to the default `11434` and fail to connect.
- **Purdue GenAI Studio** (`genai.rcac.purdue.edu`) — OpenAI-compatible endpoint, zero
  server management. Needs `pip install langchain-openai`. Put your GenAI Studio API key
  in `.env` as `OPENAI_API_KEY=...`. Verified working end-to-end (plain chat and tool
  calling both forwarded correctly by the endpoint).
- **Anthropic (Claude)** — put your key in `.env` as `ANTHROPIC_API_KEY`. Needs
  `pip install langchain-anthropic`. Very reliable tool calling.
- **OpenAI** — put your key in `.env` as `OPENAI_API_KEY`. Needs
  `pip install langchain-openai`. Very reliable tool calling. Don't combine with the
  Purdue GenAI preset above — both reuse the `OPENAI_API_KEY` name for different services.

**Gilbreth + Ollama across notebooks:** if you already ran the Ollama install steps
above in a terminal opened from inside Jupyter, `ollama serve` is reachable from any
other notebook's kernel too (including `Workshop2_Part2b_Agent_hardgraph.ipynb`) --
just make sure each notebook's `LLM_BASE_URL` matches the port you started it on.

In [ ]:
# --- Edit these directly, no env vars needed. Uncomment ONE preset. ---

LOCAL_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # used only when LLM_MODEL is None (the local preset)

# 1) Local HF model (default) -- [local] runs anywhere (Mac or Gilbreth), no server or
#    API key, but weakest tool-calling reliability.
LLM_MODEL    = None
LLM_BASE_URL = None

# 2) Ollama -- [api, local server] best free tool-calling. See C2 above for the
#    Gilbreth no-sudo install steps.
#LLM_MODEL    = "ollama:llama3.2"
#LLM_BASE_URL = None   # set to "http://127.0.0.1:11435" if you started Ollama on that
                       # port instead, per C2's "address already in use" note

# 3) Purdue GenAI Studio -- [api] OpenAI-compatible endpoint, zero server management.
#    Needs: pip install langchain-openai. Put your GenAI Studio API key in .env as
#    OPENAI_API_KEY=... See C2 above for details.
#LLM_MODEL    = "openai:llama3.2:latest"                    # or another model from your GenAI Studio account. adding openai prefix to the model name is required for GenAI Studio, because it is an OpenAI-compatible endpoint.
#LLM_BASE_URL = "https://genai.rcac.purdue.edu/api"          # client appends /chat/completions itself

# 4) Anthropic (Claude) -- [api] very reliable tool calling. Needs:
#    pip install langchain-anthropic. Put your key in .env as ANTHROPIC_API_KEY.
# LLM_MODEL    = "anthropic:claude-opus-4-8"                 # or "anthropic:claude-sonnet-5" / "anthropic:claude-haiku-4-5" for cheaper/faster
# LLM_BASE_URL = None

# 5) OpenAI -- [api] very reliable tool calling. Needs: pip install langchain-openai.
#    Put your key in .env as OPENAI_API_KEY. Don't combine with the Purdue GenAI preset
#    above -- both reuse the OPENAI_API_KEY name for different services.
# LLM_MODEL    = "openai:gpt-4o-mini"                        # or "openai:gpt-4o" for a stronger model
# LLM_BASE_URL = None


def create_llm():
    """
    Local Hugging Face model by default. Set LLM_MODEL above to a non-None value to
    route through init_chat_model() instead -- Ollama, Purdue GenAI, Anthropic, OpenAI,
    or any OpenAI-compatible endpoint.
    """
    if LLM_MODEL is None:
        text_gen = pipeline("text-generation", model=LOCAL_MODEL, max_new_tokens=512)
        return ChatHuggingFace(llm=HuggingFacePipeline(pipeline=text_gen))
    else:
        kwargs = {"temperature": 0}
        if LLM_BASE_URL:
            kwargs["base_url"] = LLM_BASE_URL
        return init_chat_model(LLM_MODEL, **kwargs)


print("LLM backend: ", f"local ({LOCAL_MODEL})" if LLM_MODEL is None else f"api ({LLM_MODEL})")

LLM backend:  api (ollama:llama3.2)


## C3. AgentState -- the shared message history

Back to the message-list shape (compare with hardgraph version), because this is a
true multi-turn loop: every tool call and tool result needs to accumulate in one growing
history that the model re-reads on each turn. `add_messages` is the reducer that makes
new messages append instead of overwrite.

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


## C4. Build the LangGraph agent graph (generic ReAct loop)

```
START → agent ──┬─(tool_calls?)→ tools → agent (loop)
                └─(no tool calls)→ END
```

`agent_node` binds **every** discovered tool at once and invokes the model on the full
history; `should_continue` routes to `tools` whenever the last message has `tool_calls`,
otherwise ends. Nothing here restricts which tool the model can call on which turn --
the model is free to call `create_notification` first, `search_knowledge_base` first, or
skip a lookup entirely. Whether it does the right thing depends entirely on the system
prompt (C5) and the model's own reliability -- that's the whole point of this notebook.

In [5]:
def build_graph(tools):
    llm            = create_llm()
    # bind_tools attaches every tool's schema to every LLM call -- the model decides,
    # turn by turn, whether and which tool to call. No gating, no fixed order.
    llm_with_tools = llm.bind_tools(tools)

    def agent_node(state: AgentState) -> dict:
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    def should_continue(state: AgentState) -> str:
        last_message = state["messages"][-1]
        if last_message.tool_calls:
            return "tools"
        return END

    tool_node = ToolNode(tools)

    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tool_node)
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    graph.add_edge("tools", "agent")   # after tool execution, go back to agent
    return graph.compile()


## C5. System prompt -- this is the only thing enforcing order

This is a fully-worked-out prompt attempt (explicit ordering rules, an explicit ban on
placeholder text, and an explicit "re-call it if you already wrote it wrong"
instruction) -- a genuine best effort at solving this with prompting alone, not a
strawman. The alternative notebook, `Workshop2_Part2b_Agent_hardgraph.ipynb`, uses the
same ideas but doesn't need to rely on the model following them, because its graph
structure enforces the order instead. Edit this prompt and re-run C5 + C6/C7 to try your
own wording.

In [6]:
SYSTEM_PROMPT = """
You are the Boilermaker Autonomous TA for a Purdue course.
Your job is to help students and instructors manage course planning using the tools available.

Assume today's date is September 1, 2026 -- the course is partway through its Fall
semester. Use this as "today" whenever the question is relative (e.g. "this week,"
"coming up," "next two weeks"), and when you propose a date for a new event, it must
fall within this Fall semester (not the following Spring) and must not be in the past.

Tools:
- search_knowledge_base(query): search the course knowledge base for syllabus details, policies, and study guidance.
- get_academic_calendar(query): read academic calendar events.
- create_notification(subject, body): write a formatted announcement to the announcements file.

Always use the tools when you need facts from the knowledge base or calendar.

Follow this order strictly:
1. First, call search_knowledge_base and/or get_academic_calendar as many times as needed
   to gather every specific fact the announcement will require (dates, times, policies).
   Do this before writing anything.
2. If the question asks about an existing fact (a policy, an exam date, a deadline) and
   the tool results don't contain it, say so plainly instead of guessing.
3. Some questions instead ask you to schedule a NEW event (e.g. a review session) that
   won't appear in any tool result, because it doesn't exist yet -- you have to invent
   it. In that case, pick a specific date and time yourself, using the retrieved
   academic calendar as constraints (avoid any date/time that conflicts with a listed
   holiday, exam, or deadline), and state your choice plainly as your own proposal.
4. Only after you have those results, call create_notification with every detail filled
   in -- using retrieved facts where they exist, or your own concrete proposed
   date/time for a new event you were asked to schedule. Never write placeholder text
   like "[insert date]" or "[insert time]" -- always commit to a real, specific value
   rather than leaving a gap.
5. create_notification writes to the announcements file immediately when called, and the
   file only reflects your most recent call -- it is not updated automatically to match
   whatever you say afterward. So if you already called create_notification earlier in
   this conversation and its content contains a placeholder or a value you have since
   corrected, you must call create_notification again with the corrected text as your
   final tool call before finishing.
"""


## C6. Connect to the MCP server and run the agent

In [7]:
DEFAULT_QUESTION = (
    "A professor wants to schedule a single CS course review session that does not conflict "
    "with holidays or exams. Summarize the plan and write a notification announcement for students."
)

MCP_SERVER_URL = "http://127.0.0.1:8001/mcp"


async def run_agent(question: str = DEFAULT_QUESTION):
    log.info("Connecting to MCP server at %s", MCP_SERVER_URL)

    try:
        client = MultiServerMCPClient({
            "boiler_ta": {
                "url": MCP_SERVER_URL,
                "transport": "streamable_http",
            },
        })
        tools = await client.get_tools()
    except Exception as e:
        log.error("Could not connect to MCP server: %s", e)
        return

    log.info("Loaded %d tools: %s", len(tools), [t.name for t in tools])

    agent  = build_graph(tools)
    result = await agent.ainvoke({
        "messages": [SystemMessage(content=SYSTEM_PROMPT), HumanMessage(content=question)],
    })

    for msg in result["messages"]:
        if isinstance(msg, AIMessage) and msg.content:
            print("\nAgent reply:\n", msg.content)


# Run the async agent inside the notebook
await run_agent()


13:57:01  INFO      Connecting to MCP server at http://127.0.0.1:8001/mcp
13:57:01  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:57:01  INFO      Received session ID: 312b7f8413764ee2b5f0b2ac1d23f8d6
13:57:01  INFO      Negotiated protocol version: 2025-11-25
13:57:01  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 202 Accepted"
13:57:01  INFO      HTTP Request: GET http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:57:01  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:57:01  INFO      HTTP Request: DELETE http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:57:01  INFO      Loaded 3 tools: ['search_knowledge_base', 'get_academic_calendar', 'create_notification']
13:57:17  INFO      HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
13:57:18  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:57:18  INFO      Received session ID: 9b1896702b83478d82054efd48f1c724
13:57:18  INFO


Agent reply:
 A review session for the CS course has been scheduled. Please check the course website for details.


## C7. Try a custom question

Run the same question here and in `Workshop2_Part2b_Agent_hardgraph.ipynb`, then
compare C8's `announcements.txt` output between the two notebooks (clear the file
between runs for a clean comparison).

In [8]:
await run_agent("What assignment deadlines are coming up in the next two weeks? Summarize them for students.")


13:57:19  INFO      Connecting to MCP server at http://127.0.0.1:8001/mcp
13:57:19  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:57:19  INFO      Received session ID: b80a534a44d1422e89bc357e88923201
13:57:19  INFO      Negotiated protocol version: 2025-11-25
13:57:19  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 202 Accepted"
13:57:19  INFO      HTTP Request: GET http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:57:19  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:57:19  INFO      HTTP Request: DELETE http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:57:19  INFO      Loaded 3 tools: ['search_knowledge_base', 'get_academic_calendar', 'create_notification']
13:57:20  INFO      HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
13:57:20  INFO      HTTP Request: POST http://127.0.0.1:8001/mcp "HTTP/1.1 200 OK"
13:57:20  INFO      Received session ID: 068c9061612f40f38830334396eb109a
13:57:20  INFO


Agent reply:
 Here is a summary of the upcoming assignment deadlines for students:

In the next two weeks, please note the following assignment deadlines:

* October 22: Midterm 1 for CS 182
* November 5 (just before Fall Break): Last chance to submit any outstanding assignments or projects for CS 182
* November 19: Midterm 2 for CS 182

Make sure to plan your study schedule accordingly and submit any pending work on time. If you have any questions or concerns, feel free to reach out to the instructor or teaching assistant.


## C8. Check the announcements file

In [9]:
if ANNOUNCEMENTS_FILE.exists():
    print(ANNOUNCEMENTS_FILE.read_text())
else:
    print("No announcements written yet.")


Subject: Upcoming CS Course Review Session
A review session for the CS course has been scheduled. Please check the course website for details.
---



---
## Comparing this to the hardgraph alternative

A few things worth trying, to turn this into an actual experiment rather than a vibe:

- **Run the exact same `DEFAULT_QUESTION` in both notebooks**, with the same `LLM_MODEL`
  preset, clearing `announcements.txt` before each run. Does this notebook ever call
  `create_notification` before a lookup tool? Does the file ever end up with a
  placeholder while the printed "Agent reply" text looks correct (the original bug)?
- **Sweep across backends** (local HF, Ollama, and -- if you have a key -- Purdue GenAI,
  Anthropic, or OpenAI). The gap between this notebook and the hardgraph version should
  shrink as the model gets stronger, but note whether it ever fully closes.
- **Try editing the system prompt** (C5) to be even more explicit, or add a worked
  example of correct tool-call ordering. Better prompting *can* reduce the failure rate
  here -- that's real, not a strawman -- but it's still a probability, not a guarantee.
  `Workshop2_Part2b_Agent_hardgraph.ipynb`'s fixed graph doesn't need the model to get
  the order right, because there is no turn where the wrong order is even possible.

The honest takeaway: this ReAct version is more flexible (it can handle questions that
need a different sequence of tools, or no notification at all, without any code change),
but its correctness depends on prompt quality and model capability.
`Workshop2_Part2b_Agent_hardgraph.ipynb` trades away some of that flexibility for a
guarantee -- open it next to see how the same task looks as a fixed graph over the same
tools. Which one is "better" depends on how much the task's tool order can vary -- see
the last bullet in the hardgraph notebook's Extension ideas.